<a href="https://colab.research.google.com/github/Andrew-Ayegh/Dual-Stream-Prostate-Cancer-Detection/blob/main/Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ---------------IMPORT AND INSTALL LIBRARIES -------------------------#
!pip install picai-baseline SimpleITK captum scikit-learn seaborn -q
!pip install picai_prep -q
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import SimpleITK as sitk
import torch
import torchvision
from torch import nn
import sklearn
import json
from pprint import pprint
import SimpleITK as sitk
import numpy as np
import picai_prep
import picai_prep.preprocessing as pp
from picai_prep.preprocessing import PreprocessingSettings, Sample
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import torchvision.transforms.functional as TF
import random
from PIL import Image
from collections import Counter
import sys
import time
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from tqdm import tqdm


#---------- Mount Google Drive-----------#
from google.colab import drive
drive.mount('/content/drive')


# --------Verify if GPU is being used------------#
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print("Setup complete")


#--------- Define device first so everything below can use it
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Operational hardware gateway: {device}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 907.8/907.8 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 MB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.2/455.2 kB 37.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.0/43.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.1/56.1 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 89.1 MB/s eta 0:00:00
If you have questions or suggestions, feel free to open an issue at https://github.com/DIAGNijmegen/picai_prep

Mounted at /content/drive
PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
Setup complete
Operational hardware gateway: cuda


In [2]:
with open ('/content/drive/MyDrive/PICAI/labeled_dataset.json', 'r') as f:
    dataset = json.load(f)

malignant = sum(1 for data in dataset if data['diagnosis'] == 1)
benign = len(dataset) - malignant

In [3]:

#----------------COMPUTE ADC DATASET-WISE STATISTICS
#---------------Incremental computation becuase it chaches alot of files
# --------------Processes one patient at a time to avoid
# ---------------RAM crash. Uses two passes. first for mean, second for std.

def compute_adc_stats(samples, preprocessed_folder):
    """Compute mean and std of ADC values incrementally to avoid RAM overflow"""

    # ---Compute mean
    total_sum = 0.0
    total_count = 0

    # print("Computing ADC mean...")
    for i, patient in enumerate(samples):
        npz_path = os.path.join(preprocessed_folder, f"{patient['patient_id']}.npz")
        data = np.load(npz_path)
        adc = data['adc'].astype(np.float32)

        #----------Clip outliers
        low = np.percentile(adc, 0.5)
        high = np.percentile(adc, 99.5)
        adc = np.clip(adc, low, high)

        total_sum += adc.sum()
        total_count += adc.size

        #--------Explicitly delete from memory after each patient
        del adc, data

        if (i + 1) % 100 == 0:
            print(f"  {i + 1}/{len(samples)} patients processed")

    adc_mean = total_sum / total_count
    print(f"ADC mean: {adc_mean:.4f}")

    # --------Compute std
    total_sq_diff = 0.0

    # print("Computing ADC std...")
    for i, patient in enumerate(samples):
        npz_path = os.path.join(preprocessed_folder, f"{patient['patient_id']}.npz")
        data = np.load(npz_path)
        adc = data['adc'].astype(np.float32)

        #----------Same clipping
        low = np.percentile(adc, 0.5)
        high = np.percentile(adc, 99.5)
        adc = np.clip(adc, low, high)

        total_sq_diff += ((adc - adc_mean) ** 2).sum()

        #----------Explicitly delete from memory after each patient
        del adc, data

        if (i + 1) % 100 == 0:
            print(f"  {i + 1}/{len(samples)} patients processed")

    adc_std = np.sqrt(total_sq_diff / total_count)
    print(f"ADC std: {adc_std:.4f}")

    return float(adc_mean), float(adc_std)





#----------------------STRATIFIED SPLIT
def stratified_split(dataset):
    """Split dataset into train/val/test maintaining class distribution"""

    malignant = [p for p in dataset if p['diagnosis'] == 1]
    benign = [p for p in dataset if p['diagnosis'] == 0]

    print(f"Total malignant: {len(malignant)}")
    print(f"Total benign: {len(benign)}")

    mal_train, mal_temp = train_test_split(malignant, test_size=0.30, random_state=42)
    mal_val, mal_test = train_test_split(mal_temp, test_size=0.333, random_state=42)

    ben_train, ben_temp = train_test_split(benign, test_size=0.30, random_state=42)
    ben_val, ben_test = train_test_split(ben_temp, test_size=0.333, random_state=42)

    train_set = mal_train + ben_train
    val_set = mal_val + ben_val
    test_set = mal_test + ben_test

    random.shuffle(train_set)
    random.shuffle(val_set)
    random.shuffle(test_set)

    print(f"\nTrain: {len(train_set)} patients")
    print(f"Malignant: {sum(1 for p in train_set if p['diagnosis']==1)}")
    print(f"Benign: {sum(1 for p in train_set if p['diagnosis']==0)}")

    print(f"\nValidation: {len(val_set)} patients")
    print(f"Malignant: {sum(1 for p in val_set if p['diagnosis']==1)}")
    print(f"Benign: {sum(1 for p in val_set if p['diagnosis']==0)}")

    print(f"\nTest: {len(test_set)} patients")
    print(f"Malignant: {sum(1 for p in test_set if p['diagnosis']==1)}")
    print(f"Benign: {sum(1 for p in test_set if p['diagnosis']==0)}")

    return train_set, val_set, test_set






#-----------------------AUGMENTATION
def augment_slice(t2w_slice, adc_slice):
    """Apply random augmentation to a pair of 2D slices BUT only for training data"""

    t2w_pil = TF.to_pil_image(t2w_slice.astype(np.float32))
    adc_pil = TF.to_pil_image(adc_slice.astype(np.float32))

    if random.random() > 0.5:
        t2w_pil = TF.hflip(t2w_pil)
        adc_pil = TF.hflip(adc_pil)

    if random.random() > 0.5:
        t2w_pil = TF.vflip(t2w_pil)
        adc_pil = TF.vflip(adc_pil)

    angle = random.uniform(-15, 15)
    t2w_pil = TF.rotate(t2w_pil, angle)
    adc_pil = TF.rotate(adc_pil, angle)

    brightness_factor = random.uniform(0.9, 1.1)
    contrast_factor = random.uniform(0.9, 1.1)
    t2w_pil = TF.adjust_brightness(t2w_pil, brightness_factor)
    t2w_pil = TF.adjust_contrast(t2w_pil, contrast_factor)
    # ADC are not regular images there are water molecule representation so, adding brightness and contrast to it, might be against the systems wellbeing
    # adc_pil = TF.adjust_brightness(adc_pil, brightness_factor)
    # adc_pil = TF.adjust_contrast(adc_pil, contrast_factor)

    t2w_aug = np.array(t2w_pil)
    adc_aug = np.array(adc_pil)

    return t2w_aug, adc_aug



#--------------------------------NORMALISATION FUNCTION
def normalise_t2w(slice_array):
    """instance-wise z-score normalization for T2W slices."""
    low = np.percentile(slice_array, 0.5)
    high = np.percentile(slice_array, 99.5)

    if (high - low) < 1e-3:
        return np.zeros_like(slice_array)

    clipped = np.clip(slice_array, low, high)
    mean = clipped.mean()
    std = clipped.std()

    if std < 1e-2:
        return np.zeros_like(slice_array)

    epsilon = 1e-8
    return (clipped - mean) / (std + epsilon)


def normalise_adc(slice_array, adc_mean, adc_std):
    """Dataset-wise z-score normalization for ADC slices."""
    low = np.percentile(slice_array, 0.5)
    high = np.percentile(slice_array, 99.5)

    if (high - low) < 1e-3:
        return np.zeros_like(slice_array)

    clipped = np.clip(slice_array, low, high)

    if clipped.std() < 1e-2:
        return np.zeros_like(slice_array)

    epsilon = 1e-8
    return (clipped - adc_mean) / (adc_std + epsilon)



#------------------ RESIZE FUNCTION
def resize_slice(slice_array, target_size=(224, 224)):
    """Resize a 2D slice using pure PyTorch tensors to preserve true physical float scales."""
    #------------Convert numpy array to 4D tensor: [Batch=1, Channel=1, Height, Width]
    tensor = torch.tensor(slice_array, dtype=torch.float32).unsqueeze(0).unsqueeze(0)

    #-----------Bilinear resize without scaling values
    resized = nn.functional.interpolate(tensor, size=target_size, mode='bilinear', align_corners=False)

    #---------Return back as a 2D numpy array
    return resized.squeeze(0).squeeze(0).numpy()




In [4]:
#---------------------- DATASET CLASS
class ProstateDataset(Dataset):
    def __init__(self, samples, preprocessed_folder, mode='train',
             adc_mean=None, adc_std=None, prebuilt_slices=None):
      self.preprocessed_folder = preprocessed_folder
      self.mode = mode
      self.adc_mean = adc_mean
      self.adc_std = adc_std

      if prebuilt_slices is not None:
          #----------Use pre-saved slice list
          self.valid_slices = prebuilt_slices
          print(f"  {mode} dataset: loaded {len(self.valid_slices)} pre-built slices")
      else:
          #----------Build from scratch
          print(f"  Building {mode} valid slice list...")
          self.valid_slices = []
          for patient in samples:
              npz_path = os.path.join(preprocessed_folder, f"{patient['patient_id']}.npz")
              data = np.load(npz_path)
              for slice_idx in range(20):
                  t2w_slice = data['t2w'][slice_idx]
                  if len(np.unique(t2w_slice)) > 10:
                      self.valid_slices.append({
                          'patient_id': patient['patient_id'],
                          'slice_idx': slice_idx,
                          'diagnosis': patient['diagnosis']
                      })
              del data
          print(f"  {mode} dataset: built {len(self.valid_slices)} valid slices")

    def __len__(self):
        return len(self.valid_slices)

    def __getitem__(self, index):
        #--------Get the specific valid slice
        slice_info = self.valid_slices[index]
        patient_id = slice_info['patient_id']
        slice_idx = slice_info['slice_idx']
        label = slice_info['diagnosis']

        # --------Load npz file
        npz_path = os.path.join(self.preprocessed_folder, f"{patient_id}.npz")
        data = np.load(npz_path)

        #---------Extract slice
        t2w_slice = data['t2w'][slice_idx].astype(np.float32)
        adc_slice = data['adc'][slice_idx].astype(np.float32)

        # --------Resize slice (output data in a clean 0-255 range)
        t2w_slice = resize_slice(t2w_slice)
        adc_slice = resize_slice(adc_slice)

        # -------Augment BEFORE Normalization (for malignant and benign training cases)
        if self.mode == 'train':
            t2w_slice, adc_slice = augment_slice(t2w_slice, adc_slice)

        # --------Normalise data to be uniformly scaled to z-scores
        t2w_slice = normalise_t2w(t2w_slice)
        adc_slice = normalise_adc(adc_slice, self.adc_mean, self.adc_std)

        # -------clean-up blank slice
        t2w_slice = np.nan_to_num(t2w_slice, nan=0.0, posinf=0.0, neginf=0.0)
        adc_slice = np.nan_to_num(adc_slice, nan=0.0, posinf=0.0, neginf=0.0)

        # Convert to 3 channels by repeating — matches ImageNet pretrained input format
        t2w_tensor = torch.tensor(t2w_slice, dtype=torch.float32).unsqueeze(0).repeat(3, 1, 1)
        adc_tensor = torch.tensor(adc_slice, dtype=torch.float32).unsqueeze(0).repeat(3, 1, 1)
        label_tensor = torch.tensor(label, dtype=torch.float32)

        return t2w_tensor, adc_tensor, label_tensor

In [5]:
PREPROCESSED_FOLDER = '/content/drive/MyDrive/PICAI/preprocessed/'
CACHE_FOLDER = '/content/drive/MyDrive/PICAI/cache/'
os.makedirs(CACHE_FOLDER, exist_ok=True)


# -----------------Spliting data into train, validate and test set
splits_path = os.path.join(CACHE_FOLDER, 'splits.json')

if os.path.exists(splits_path):
    print("Loading saved splits...")
    with open(splits_path, 'r') as f:
        splits = json.load(f)
    train_set = splits['train']
    val_set = splits['val']
    test_set = splits['test']
    print(f"Loaded — Train: {len(train_set)} | Val: {len(val_set)} | Test: {len(test_set)}")
else:
    print("Computing splits for first time...")
    with open('/content/drive/MyDrive/PICAI/labeled_dataset.json', 'r') as f:
        dataset = json.load(f)
    train_set, val_set, test_set = stratified_split(dataset)
    with open(splits_path, 'w') as f:
        json.dump({'train': train_set, 'val': val_set, 'test': test_set}, f)
    print("Splits saved")


# -------------ADC Stats
adc_stats_path = os.path.join(CACHE_FOLDER, 'adc_stats.json')

if os.path.exists(adc_stats_path):
    print("\nLoading saved ADC stats...")
    with open(adc_stats_path, 'r') as f:
        adc_stats = json.load(f)
    adc_mean = adc_stats['adc_mean']
    adc_std = adc_stats['adc_std']
    print(f"Loaded — ADC mean: {adc_mean:.4f} | ADC std: {adc_std:.4f}")
else:
    print("\nComputing ADC stats for first time...")
    adc_mean, adc_std = compute_adc_stats(train_set, PREPROCESSED_FOLDER)
    with open(adc_stats_path, 'w') as f:
        json.dump({'adc_mean': adc_mean, 'adc_std': adc_std}, f)
    print("ADC stats saved")

#---------------valid slice list
train_slices_path = os.path.join(CACHE_FOLDER, 'train_valid_slices.json')
val_slices_path = os.path.join(CACHE_FOLDER, 'val_valid_slices.json')
test_slices_path = os.path.join(CACHE_FOLDER, 'test_valid_slices.json')

if os.path.exists(train_slices_path):
    print("\nLoading saved valid slice lists...")
    with open(train_slices_path, 'r') as f:
        train_valid_slices = json.load(f)
    with open(val_slices_path, 'r') as f:
        val_valid_slices = json.load(f)
    with open(test_slices_path, 'r') as f:
        test_valid_slices = json.load(f)
    print(f"Loaded — Train: {len(train_valid_slices)} | Val: {len(val_valid_slices)} | Test: {len(test_valid_slices)} slices")
else:
    print("\nBuilding valid slice lists for first time...")
    #-----------Building datasets to generate valid slice lists
    train_dataset_temp = ProstateDataset(train_set, PREPROCESSED_FOLDER, mode='train', adc_mean=adc_mean, adc_std=adc_std)
    val_dataset_temp = ProstateDataset(val_set, PREPROCESSED_FOLDER, mode='val', adc_mean=adc_mean, adc_std=adc_std)
    test_dataset_temp = ProstateDataset(test_set, PREPROCESSED_FOLDER, mode='test', adc_mean=adc_mean, adc_std=adc_std)

    train_valid_slices = train_dataset_temp.valid_slices
    val_valid_slices = val_dataset_temp.valid_slices
    test_valid_slices = test_dataset_temp.valid_slices

    with open(train_slices_path, 'w') as f:
        json.dump(train_valid_slices, f)
    with open(val_slices_path, 'w') as f:
        json.dump(val_valid_slices, f)
    with open(test_slices_path, 'w') as f:
        json.dump(test_valid_slices, f)
    print("Valid slice lists saved")


# ------creating dataset from cached slicrs in drive

print("\nCreating datasets from cached data...")

train_dataset = ProstateDataset(
    train_set, PREPROCESSED_FOLDER,
    mode='train', adc_mean=adc_mean, adc_std=adc_std,
    prebuilt_slices=train_valid_slices
)
val_dataset = ProstateDataset(
    val_set, PREPROCESSED_FOLDER,
    mode='val', adc_mean=adc_mean, adc_std=adc_std,
    prebuilt_slices=val_valid_slices
)
test_dataset = ProstateDataset(
    test_set, PREPROCESSED_FOLDER,
    mode='test', adc_mean=adc_mean, adc_std=adc_std,
    prebuilt_slices=test_valid_slices
)

print(f"Train dataset: {len(train_dataset)} slices")
print(f"Val dataset:   {len(val_dataset)} slices")
print(f"Test dataset:  {len(test_dataset)} slices")


# --------------DATALOADERS

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=0, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, num_workers=0, pin_memory=True)

print("\nDataLoaders created successfully")
print("\nSetup complete — ready for training")

Loading saved splits...
Loaded — Train: 1032 | Val: 295 | Test: 149

Loading saved ADC stats...
Loaded — ADC mean: 805.8851 | ADC std: 834.7990

Loading saved valid slice lists...
Loaded — Train: 20485 | Val: 5862 | Test: 2954 slices

Creating datasets from cached data...
  train dataset: loaded 20485 pre-built slices
  val dataset: loaded 5862 pre-built slices
  test dataset: loaded 2954 pre-built slices
Train dataset: 20485 slices
Val dataset:   5862 slices
Test dataset:  2954 slices

DataLoaders created successfully

Setup complete — ready for training


In [6]:
marksheet_path = '/content/drive/MyDrive/PICAI/labels/picai_labels-main/clinical_information/marksheet.csv'
df = pd.read_csv(marksheet_path)
print(df.columns.tolist())
print(df.head(10))
print(f"\nTotal rows: {len(df)}")
print(f"\nValue counts for csPCa column:")
print(df['case_csPCa'].value_counts())

['patient_id', 'study_id', 'mri_date', 'patient_age', 'psa', 'psad', 'prostate_volume', 'histopath_type', 'lesion_GS', 'lesion_ISUP', 'case_ISUP', 'case_csPCa', 'center']
   patient_id  study_id    mri_date  patient_age    psa  psad  \
0       10000   1000000  2019-07-02           73   7.70   NaN   
1       10001   1000001  2016-05-27           64   8.70  0.09   
2       10002   1000002  2021-04-18           58   4.20  0.06   
3       10003   1000003  2019-04-05           72  13.00   NaN   
4       10004   1000004  2020-10-21           67   8.00  0.10   
5       10005   1000005  2012-07-18           64  12.10  0.24   
6       10006   1000006  2020-10-23           73   6.20  0.23   
7       10007   1000007  2020-10-31           68   3.83  0.09   
8       10008   1000008  2020-12-06           81  11.10  0.20   
9       10009   1000009  2017-11-02           65  24.00   NaN   

   prostate_volume histopath_type lesion_GS lesion_ISUP  case_ISUP case_csPCa  \
0             55.0           MRB

In [7]:

with open('/content/drive/MyDrive/PICAI/train_valid_slices.json', 'r') as f:
    train_valid_slices = json.load(f)

In [8]:
import torchvision.models as models

def build_resnet50_stream():
    #-----------------Load pretrained ResNet-50
    resnet = models.resnet50(pretrained=True)

    #----------Freeze first two stages (layer1 and layer2)
    for param in resnet.layer1.parameters():
        param.requires_grad = False
    for param in resnet.layer2.parameters():
        param.requires_grad = False

    #----------Remove final avgpool and fc layers
    modules = list(resnet.children())[:-2]
    stream = nn.Sequential(*modules)

    return stream

In [9]:
def build_densenet121_stream():
    #----------Load pretrained DenseNet-121
    densenet = models.densenet121(pretrained=True)

    #--------Freeze first two dense blocks
    for param in densenet.features.denseblock1.parameters():
        param.requires_grad = False
    for param in densenet.features.denseblock2.parameters():
        param.requires_grad = False

    #-----------Remove classifier and keep only features
    stream = densenet.features

    return stream

In [10]:
class ChannelAttention(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.mlp = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(),
            nn.Linear(channels // reduction, channels, bias=False)
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg = self.avg_pool(x).view(x.size(0), -1)
        max_ = self.max_pool(x).view(x.size(0), -1)
        avg_out = self.mlp(avg)
        max_out = self.mlp(max_)
        scale = self.sigmoid(avg_out + max_out).view(x.size(0), x.size(1), 1, 1)
        return x * scale

class SpatialAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size=7, padding=3, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg = torch.mean(x, dim=1, keepdim=True)
        max_, _ = torch.max(x, dim=1, keepdim=True)
        combined = torch.cat([avg, max_], dim=1)
        scale = self.sigmoid(self.conv(combined))
        return x * scale

class CBAM(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.channel_attention = ChannelAttention(channels, reduction)
        self.spatial_attention = SpatialAttention()

    def forward(self, x):
        x = self.channel_attention(x)
        x = self.spatial_attention(x)
        return x

In [11]:
class ClassificationHead(nn.Module):
    def __init__(self, input_channels=3072):
        super().__init__()
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.fc1 = nn.Linear(input_channels, 256)
        self.bn = nn.BatchNorm1d(256)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(p=0.5)
        self.fc2 = nn.Linear(256, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.gap(x)
        x = x.view(x.size(0), -1)
        x = self.relu(self.bn(self.fc1(x)))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

In [12]:
class DualStreamProstateCancerDetector(nn.Module):
    def __init__(self):
        super().__init__()

        self.dropout = nn.Dropout(p=0.5)

        # -------------Structural dropout for individual backbones
        self.stream_dropout = nn.Dropout(p=0.3)

        #-----------Stream A ResNet-50 for T2W
        self.resnet_stream = build_resnet50_stream()

        #-----------Stream B DenseNet-121 for ADC
        self.densenet_stream = build_densenet121_stream()

        #----------CBAM fusion  2048 + 1024 = 3072 channels
        self.cbam = CBAM(channels=3072, reduction=16)

        #----------Classification head
        self.classifier = ClassificationHead(input_channels=3072)

    def forward(self, t2w, adc):
        #---------------Stream A processes T2W
        t2w_features = self.resnet_stream(t2w)
        t2w_features = self.stream_dropout(t2w_features)

        #----------Stream B processes ADC
        adc_features = self.densenet_stream(adc)


        #-------------Add batch norm after DenseNet
        adc_features = nn.functional.relu(adc_features)
        adc_features = self.stream_dropout(adc_features)

        #--------------Resize to same spatial size
        if t2w_features.shape[2:] != adc_features.shape[2:]:
            adc_features = nn.functional.interpolate(
                adc_features,
                size=t2w_features.shape[2:],
                mode='bilinear',
                align_corners=False
            )

        #------------Concatenate along channel dimension
        fused = torch.cat([t2w_features, adc_features], dim=1)

        #-----------Applying CBAM attention
        fused = self.cbam(fused)

        fused = self.dropout(fused)

        #------Classify
        output = self.classifier(fused)

        return output

In [13]:
#--------------Class weights for imbalanced dataset
n_total = len(train_set)
n_malignant = sum(1 for p in train_set if p['diagnosis'] == 1)
n_benign = sum(1 for p in train_set if p['diagnosis'] == 0)
n_classes = 2

weight_malignant = n_total / (n_classes * n_malignant)
weight_benign = n_total / (n_classes * n_benign)

print(f"Malignant weight: {weight_malignant:.4f}")
print(f"Benign weight: {weight_benign:.4f}")

pos_weight = torch.tensor([weight_malignant / weight_benign]).to(device)

Malignant weight: 1.7374
Benign weight: 0.7020


In [14]:
#--------------Instantiate the model and psuh to GPU
model = DualStreamProstateCancerDetector().to(device)


#-----Verify structure size
total_params = sum(p.numel() for p in model.parameters())
print(f"Total model parameters loaded: {total_params:,}")

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 186MB/s]
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=DenseNet121_Weights.IMAGENET1K_V1`. You can also use `weights=DenseNet121_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:00<00:00, 139MB/s]


Total model parameters loaded: 32,429,091


In [15]:
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

# had to change the decay to 1e-3 because of the consistent model ovefitting the overpenalizing
# optimizer = optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)
optimizer = optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-3)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=5)

In [16]:
def train_epoch(model, loader, criterion, optimizer, device, scaler):
    model.train()
    total_loss, correct, total = 0, 0, 0
    pbar = tqdm(loader, desc="   Training (GPU-AMP)", file=sys.stdout, leave=True)
    for t2w, adc, labels in pbar:
        t2w, adc, labels = t2w.to(device), adc.to(device), labels.to(device).unsqueeze(1)
        optimizer.zero_grad()

        with torch.cuda.amp.autocast():
            outputs = model(t2w, adc)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        predicted = (torch.sigmoid(outputs) >= 0.5).float()
        correct += (predicted == labels).sum().item()
        total += labels.size(0)
        pbar.set_postfix(loss=f"{loss.item():.4f}", acc=f"{(correct/total):.4f}")
    return total_loss / len(loader), correct / total

def validate_epoch(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    pbar = tqdm(loader, desc="   Validating", file=sys.stdout, leave=True)
    with torch.no_grad():
        for t2w, adc, labels in pbar:
            t2w, adc, labels = t2w.to(device), adc.to(device), labels.to(device).unsqueeze(1)
            with torch.cuda.amp.autocast():
                outputs = model(t2w, adc)
                loss = criterion(outputs, labels)
            total_loss += loss.item()
            predicted = (torch.sigmoid(outputs) >= 0.5).float()
            correct += (predicted == labels).sum().item()
            total += labels.size(0)
            pbar.set_postfix(loss=f"{loss.item():.4f}", acc=f"{(correct/total):.4f}")
    return total_loss / len(loader), correct / total

In [17]:
def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, device, epochs=50, patience=10, start_epoch=0, history=None):
    best_val_loss = float('inf')
    patience_counter = 0
    best_model_path = '/content/drive/MyDrive/PICAI/checkpoints/best_model.pth'
    latest_checkpoint_path = '/content/drive/MyDrive/PICAI/checkpoints/latest_checkpoint.pth'
    os.makedirs('/content/drive/MyDrive/PICAI/checkpoints/', exist_ok=True)

    scaler = torch.cuda.amp.GradScaler()

    if history is None:
        history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    else:
        print(f"  -> Continuing with existing history tracking ({len(history['train_loss'])} epochs recorded)")
        if len(history['val_loss']) > 0:
            best_val_loss = min(history['val_loss'])
            min_idx = history['val_loss'].index(best_val_loss)
            patience_counter = len(history['val_loss']) - 1 - min_idx

    for epoch in range(start_epoch, epochs):
        train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device, scaler)
        val_loss, val_acc = validate_epoch(model, val_loader, criterion, device)
        scheduler.step(val_loss)

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)

        print(f"Epoch {epoch+1}/{epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

        if (epoch + 1) % 1 == 0:
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict(),
                'val_loss': val_loss,
                'history': history
            }, latest_checkpoint_path)
            print(f"   [Checkpoint] Progress saved to Drive at epoch {epoch+1}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(), 'optimizer_state_dict': optimizer.state_dict(), 'val_loss': val_loss}, best_model_path)
            print(f"   ✓ Best model saved — val loss: {val_loss:.4f}")
        else:
            patience_counter += 1
            print(f"   No improvement — early stopping patience: {patience_counter}/{patience}")

        with open('/content/drive/MyDrive/PICAI/checkpoints/history.json', 'w') as f:
            json.dump(history, f)

        if patience_counter >= patience:
            print(f"Early stopping triggered at epoch {epoch+1}")
            break
    return history

In [ ]:

#------------RESUME CHECKPOINT BOOTSTRAPPER
start_epoch = 0
history = None
checkpoint_path = '/content/drive/MyDrive/PICAI/checkpoints/latest_checkpoint.pth'

if os.path.exists(checkpoint_path):
    print("=+= Found existing progress checkpoint! Resuming execution...")
    #-----------Load state dictionaries directly into memory
    checkpoint = torch.load(checkpoint_path, map_location=device)

    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    scheduler.load_state_dict(checkpoint['scheduler_state_dict'])

    #----------Set the loop index to pick up from where it stopped
    start_epoch = checkpoint['epoch'] + 1
    history = checkpoint['history']
    print(f"=+= Success! Ready to resume training from Epoch {start_epoch+1}")
else:
    print("=+= No tracking checkpoint found. Preparing a clean, fresh training initialization...")

#-------And the Training process begins
history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    device=device,
    epochs=50,
    patience=10,
    start_epoch=start_epoch, # Passes the dynamic start gate
    history=history          # Passes the loaded progress dictionary
)

/tmp/ipykernel_5182/465286215.py:8: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


=+= No tracking checkpoint found. Preparing a clean, fresh training initialization...
   Training (GPU-AMP):   0%|          | 0/1281 [00:00<?, ?it/s]

/tmp/ipykernel_5182/3636700368.py:9: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


   Validating:   0%|          | 1/367 [00:01<09:17,  1.52s/it, acc=0.0000, loss=3.0873]

/tmp/ipykernel_5182/3636700368.py:31: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


   Validating: 100%|██████████| 367/367 [09:04<00:00,  1.48s/it, acc=0.2880, loss=1.5600]
Epoch 1/50 | Train Loss: 1.0169 | Val Loss: 1.7937
   [Checkpoint] Progress saved to Drive at epoch 1
   ✓ Best model saved — val loss: 1.7937
   Validating: 100%|██████████| 367/367 [05:16<00:00,  1.16it/s, acc=0.2878, loss=0.9941]
Epoch 2/50 | Train Loss: 0.9954 | Val Loss: 1.0497
   [Checkpoint] Progress saved to Drive at epoch 2
   ✓ Best model saved — val loss: 1.0497
   Validating: 100%|██████████| 367/367 [05:21<00:00,  1.14it/s, acc=0.2878, loss=0.8917]
Epoch 3/50 | Train Loss: 0.9853 | Val Loss: 1.0031
   [Checkpoint] Progress saved to Drive at epoch 3
   ✓ Best model saved — val loss: 1.0031
   Validating: 100%|██████████| 367/367 [05:19<00:00,  1.15it/s, acc=0.7059, loss=0.3169]
Epoch 4/50 | Train Loss: 0.9836 | Val Loss: 0.9838
   [Checkpoint] Progress saved to Drive at epoch 4
   ✓ Best model saved — val loss: 0.9838
   Validating: 100%|██████████| 367/367 [05:19<00:00,  1.15it/s, acc